# Reasoning Budgets — companion notebook

Work out what hidden **thinking tokens** cost you in money and wall-clock time,
and find the accuracy gain a reasoning tier must deliver to be worth it.

Standard library plus matplotlib. Rates are illustrative — substitute your
provider's current published numbers in the first cell.

## 1. Tiers and the task

The key asymmetry: input is prefilled in **parallel** (cheap in time), while
thinking and output are decoded **serially** (expensive in time) and billed at
the higher output rate.

In [1]:
from dataclasses import dataclass

PREFILL_TOK_PER_S = 3000.0


@dataclass
class Tier:
    name: str
    usd_in_per_m: float
    usd_out_per_m: float
    decode_tok_per_s: float
    min_thinking: int      # 0 => thinking can be switched off entirely


REASONING = Tier("reasoning", 1.25, 10.00, 100.0, min_thinking=128)
CHEAP = Tier("small", 0.10, 0.40, 250.0, min_thinking=0)

IN_TOK, OUT_TOK = 1800, 40   # the answer length never changes below


def call(tier, thinking, cached_in=0):
    thinking = max(thinking, tier.min_thinking)   # the clamp that surprises people
    fresh = IN_TOK - cached_in
    usd = (fresh * tier.usd_in_per_m + cached_in * tier.usd_in_per_m * 0.25) / 1e6
    usd += (thinking + OUT_TOK) * tier.usd_out_per_m / 1e6
    secs = IN_TOK / PREFILL_TOK_PER_S + (thinking + OUT_TOK) / tier.decode_tok_per_s
    return {"thinking": thinking, "usd": usd, "secs": secs,
            "usd_think": thinking * tier.usd_out_per_m / 1e6,
            "secs_think": thinking / tier.decode_tok_per_s}


r = call(REASONING, 0)
print(f"asked for thinking=0 on the reasoning tier -> got {r['thinking']}")
print("the parameter was accepted and ignored; only the usage counts reveal it")

asked for thinking=0 on the reasoning tier -> got 128
the parameter was accepted and ignored; only the usage counts reveal it


## 2. How much of the call is invisible?

For a fixed 40-token answer, plot the share of cost and of elapsed time that is
hidden deliberation.

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

budgets = list(range(0, 2049, 64))
rows = [call(REASONING, b) for b in budgets]
share_usd = [x["usd_think"] / x["usd"] * 100 for x in rows]
share_sec = [x["secs_think"] / x["secs"] * 100 for x in rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(budgets, share_usd, label="% of cost")
ax1.plot(budgets, share_sec, label="% of elapsed time")
ax1.axhline(50, ls="--", lw=.8, color="grey")
ax1.set_xlabel("thinking budget (tokens)")
ax1.set_ylabel("share that is invisible (%)")
ax1.set_title(f"A {OUT_TOK}-token answer")
ax1.legend()
ax1.grid(alpha=.3)

ax2.plot(budgets, [x["secs"] for x in rows], color="tab:red")
ax2.set_xlabel("thinking budget (tokens)")
ax2.set_ylabel("seconds per call")
ax2.axhspan(0, 1.0, alpha=.15, color="green")
ax2.set_title("Wall clock (green = phone-line tolerable)")
ax2.grid(alpha=.3)
plt.tight_layout()
plt.show()

obs, fix = call(REASONING, 300), call(CHEAP, 0)
print(f"reasoning, budget 300 : ${obs['usd']:.5f}  {obs['secs']:.2f}s  "
      f"({obs['secs_think']/obs['secs']*100:.0f}% invisible)")
print(f"small tier, budget 0  : ${fix['usd']:.5f}  {fix['secs']:.2f}s")
print(f"                      -> {obs['usd']/fix['usd']:.0f}x cheaper, "
      f"{obs['secs']-fix['secs']:.2f}s faster")

reasoning, budget 300 : $0.00565  4.00s  (75% invisible)
small tier, budget 0  : $0.00020  0.76s
                      -> 29x cheaper, 3.24s faster


## 3. Break-even: when is thinking worth buying?

The honest question is never "is it cheaper" — of course it is. It is **how much
extra accuracy** the expensive tier must deliver to justify the spend. Pick your
value per correct answer and read the required lift off the curve.

In [3]:
VALUE_PER_CORRECT = 0.05   # $ of value a correct answer is worth to you
BASE_ACCURACY = 0.92       # what the small tier already achieves

extra_cost = obs["usd"] - fix["usd"]
required_lift = extra_cost / VALUE_PER_CORRECT

print(f"extra cost of the reasoning tier : ${extra_cost:.5f} / call")
print(f"value of one correct answer      : ${VALUE_PER_CORRECT:.3f}")
print(f"=> it must add at least          : {required_lift*100:.2f} accuracy points")
print(f"   i.e. beat {BASE_ACCURACY*100:.0f}% by reaching "
      f"{(BASE_ACCURACY + required_lift)*100:.2f}%")

if BASE_ACCURACY + required_lift > 1.0:
    print("\n   ...which exceeds 100%: on THIS task the reasoning tier can never pay off.")
else:
    print("\n   Plausible — but you must MEASURE the lift on a scored set, not assume it.")

print("\nSweep the value per correct answer:")
print(f"{'value/correct':>14} {'required lift':>14}")
for v in (0.01, 0.05, 0.25, 1.00, 5.00):
    print(f"${v:>13.2f} {extra_cost/v*100:>13.2f}pp")

extra cost of the reasoning tier : $0.00545 / call
value of one correct answer      : $0.050
=> it must add at least          : 10.91 accuracy points
   i.e. beat 92% by reaching 102.91%

   ...which exceeds 100%: on THIS task the reasoning tier can never pay off.

Sweep the value per correct answer:
 value/correct  required lift
$         0.01         54.54pp
$         0.05         10.91pp
$         0.25          2.18pp
$         1.00          0.55pp
$         5.00          0.11pp


## 4. Stacking the levers

Tier, budget, output cap and prefix caching are **multiplicative**. Watch the
compounding, and note that caching only helps the input term — so it matters
less the more you were spending on thinking.

In [4]:
scenarios = [
    ("reasoning, no config at all",  call(REASONING, 800)),
    ("reasoning, budget 128",        call(REASONING, 0)),
    ("small tier, budget 0",         call(CHEAP, 0)),
    ("small + cache 1200 static",    call(CHEAP, 0, cached_in=1200)),
]
worst = scenarios[0][1]["usd"]
print(f"{'scenario':<32} {'$/call':>10} {'vs worst':>10} {'secs':>7}")
for label, s in scenarios:
    print(f"{label:<32} ${s['usd']:>9.5f} {worst/s['usd']:>9.0f}x {s['secs']:>6.2f}s")

print("\nAt 200 calls/day:")
for label, s in scenarios:
    print(f"  {label:<32} ${s['usd']*200*30:>8.2f} / month")

scenario                             $/call   vs worst    secs
reasoning, no config at all      $  0.01065         1x   9.00s
reasoning, budget 128            $  0.00393         3x   2.28s
small tier, budget 0             $  0.00020        54x   0.76s
small + cache 1200 static        $  0.00011       100x   0.76s

At 200 calls/day:
  reasoning, no config at all      $   63.90 / month
  reasoning, budget 128            $   23.58 / month
  small tier, budget 0             $    1.18 / month
  small + cache 1200 static        $    0.64 / month


## 5. Your turn

1. Replace the rates with your provider's current published numbers, and measure
   `decode_tok_per_s` by timing a call with a known output length.
2. Log real `thinking` counts from your own traffic — a budget is a ceiling, not
   a setting, and the distribution has a tail.
3. Re-run §3 with your measured accuracy on a scored set. If you do not have
   one, that is the actual next task — every number above is a guess without it.